In [1]:
#| default_exp _core

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
#| export 

from dataclasses import dataclass
from typing import Iterable, Sequence, Optional

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

import matplotlib.pyplot as plt

def to_device(batch, device):
    if isinstance(batch, torch.Tensor):
        return batch.to(device)
    if isinstance(batch, (list, tuple)):
        return type(batch)(to_device(x, device) for x in batch)
    if isinstance(batch, dict):
        return {k: to_device(v, device) for k, v in batch.items()}
    return batch


In [4]:
#| export

def make_tiny_dataloader(
    dataset: Dataset,
    n_items: int = 8,
    batch_size: int = 4,
    shuffle: bool = True,
) -> DataLoader:
    """
    Take the first n_items from `dataset` and build a tiny DataLoader
    for overfitting/debugging.
    """
    indices = list(range(min(n_items, len(dataset))))
    subset = torch.utils.data.Subset(dataset, indices)
    return DataLoader(subset, batch_size=batch_size, shuffle=shuffle, drop_last=True)


def batchify_video(video: torch.Tensor) -> torch.Tensor:
    """
    Ensure (b, c, t, h, w). If dataset returns (c, t, h, w), add batch dim.
    """
    if video.ndim == 4:  # (c, t, h, w)
        video = video.unsqueeze(0)
    assert video.ndim == 5, f"Expected 5D video tensor, got shape {video.shape}"
    return video

In [5]:
#| export

@dataclass
class TinyOverfitConfig:
    lr: float = 3e-4
    weight_decay: float = 0.0
    steps: int = 500
    log_every: int = 50
    mask_patches: bool = True
    max_grad_norm: Optional[float] = 1.0

def train_tiny_overfit(
    model: nn.Module,
    dataloader: DataLoader,
    cfg: TinyOverfitConfig,
    device: Optional[torch.device] = None,
):
    """
    Overfit the VideoTokenizer on a tiny dataset, logging recon / LPIPS loss.
    """
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.train()

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    step = 0
    losses = []

    while step < cfg.steps:
        for batch in dataloader:
            if isinstance(batch, (tuple, list)):
                video = batch[0]
            elif isinstance(batch, dict):
                # try some common keys
                video = batch.get("video") or batch.get("obs") or batch.get("image")
            else:
                video = batch

            video = batchify_video(video)
            video = to_device(video, device)

            opt.zero_grad(set_to_none=True)

            total_loss, tokenizer_losses = model(
                video,
                return_all_losses=True,
                mask_patches=cfg.mask_patches,
            )
            recon_loss, lpips_loss = tokenizer_losses.recon, tokenizer_losses.lpips  # ✅


            total_loss.backward()
            if cfg.max_grad_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
            opt.step()

            losses.append(
                {
                    "step": step,
                    "total": float(total_loss.detach().cpu()),
                    "recon": float(recon_loss.detach().cpu()),
                    "lpips": float(lpips_loss.detach().cpu()),
                }
            )

            if step % cfg.log_every == 0:
                print(
                    f"[{step:05d}] total={losses[-1]['total']:.4f} "
                    f"recon={losses[-1]['recon']:.4f} lpips={losses[-1]['lpips']:.4f}"
                )

            step += 1
            if step >= cfg.steps:
                break

    return losses


In [6]:
#| export

def plot_losses(losses: Sequence[dict], keys=("total", "recon", "lpips")):
    steps = [d["step"] for d in losses]
    for k in keys:
        vals = [d[k] for d in losses]
        plt.plot(steps, vals, label=k)
    plt.xlabel("step")
    plt.ylabel("loss")
    plt.legend()
    plt.title("VideoTokenizer tiny-overfit losses")
    plt.show()


In [7]:
#| export
from dreamer4.envs.pinpad import PinPad, MotionPlannerPinPad
import random

# wrap the env to return float32 images between 0 and 1
class WrappedEnv:
    def __init__(self, env):
        self.env = env
        self.observation_space = env.observation_space
        self.action_space = env.action_space

    def reset(self):
        obs = self.env.reset()
        obs = obs / 255.0
        return obs

    def step(self, action):
        obs, reward, done, trunc, info = self.env.step(action)
        obs = obs / 255.0
        return obs, reward, done, trunc, info

import cv2
def build_tiny_pinpad_dataset(device='cuda', n=1024, episode_length=8):
    env = PinPad('three', length=episode_length, extra_obs=False, size=[64, 64], random_starting_pos=True, device=device)
    mp = MotionPlannerPinPad(env)
    env = WrappedEnv(env)
    obs = env.reset()
    # print the obs stats
    # make a tiny dataset
    eps_containing_success = 0; eps = 0
    ep_contained_success = False
    videos = []; curr_video = []; total_reward = 0; ep_reward = 0
    while True:
        curr_video.append(obs)
        action = mp.sample()
        obs, reward, done, terminated, info = env.step(action); ep_reward += reward

        ep_contained_success = info['success'] or ep_contained_success
        if done or terminated:
            if ep_contained_success or random.random() > 0.5: # don't take it sometimes, HACK to get a higher success rate in the base dataset
                eps_containing_success += 1 if ep_contained_success else 0
                eps += 1
                videos.append(torch.stack(curr_video))
                total_reward += ep_reward

            # unallocate all the memory in curr_video
            del curr_video

            curr_video = []; ep_reward = 0
            ep_contained_success = False
            obs = env.reset()

        if eps >= n:
            break


    videos = torch.stack(videos)  # (n, t, c, h, w)
    videos = videos.permute(0, 2, 1, 3, 4)  # (n, c, t, h, w)

    print(f"Built tiny PinPad dataset with {videos.shape[0]} videos of shape {videos.shape[1:]} - (c, t, h, w). Success rate {eps_containing_success / eps:1.1%}")
    return torch.utils.data.TensorDataset(videos), env
    # return videos

/home/j/workspace/dreamer4/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
videos, env = build_tiny_pinpad_dataset(episode_length=100, n=100)

pads {'1', '2', '3'} target ('1', '2', '3')

Built tiny PinPad dataset with 100 videos of shape torch.Size([3, 100, 64, 64]) - (c, t, h, w). Success rate 82.0%


In [9]:
#| export

import random
from typing import Optional, Literal, Dict, Any, List
from dreamer4.envs.pinpad import PinPad, MotionPlannerPinPad

import torch
from torch.utils.data import Dataset, TensorDataset

class NormalizeObsWrapper:
    """
    Wraps a PinPad env that returns (C,H,W) uint8/float in 0..255 and
    converts observations to float32 in [0,1]. Everything else is passthrough.
    """
    def __init__(self, env):
        self.env = env
        # expose gym-ish interface
        self.action_space = env.action_space
        self.observation_space = getattr(env, "observation_space", None)
        self.device = getattr(env, "device", "cpu")
        self.size = getattr(env, "size", (64, 64))
        self.task = getattr(env, "task", "three")

    def _norm(self, obs):
        # obs is a torch.Tensor (C,H,W) from your PinPad
        obs = obs.to(torch.float32)
        # If it looks like 0..255, scale to 0..1. Otherwise assume already normalized.
        if obs.max() > 1.0 or obs.min() < 0.0:
            obs = obs / 255.0
        return obs

    def reset(self, *args, **kwargs):
        obs = self.env.reset(*args, **kwargs)
        return self._norm(obs)

    def step(self, action):
        obs, r, done, truncated, info = self.env.step(action)
        return self._norm(obs), r, done, truncated, info

    # Optional: pass through render() etc.
    def render(self, *args, **kwargs):
        return self.env.render(*args, **kwargs)

class PinPadBCEpisodes(Dataset):
    def __init__(self, env, n_episodes=512, episode_length=16, use_motion_planner=True, return_length=16):
        self.samples = []
        H, W = env.size[0], env.size[1]

        self.episode_length=episode_length; self.return_length=return_length

        if use_motion_planner:
            mp = MotionPlannerPinPad(env)

        for _ in range(n_episodes):
            frames = []
            rewards = []
            actions = []

            obs = env.reset()                  # (C, H, W) float in 0..255
            for t in range(episode_length):
                frames.append(obs.detach().cpu())              # store frame BEFORE action (standard)

                if use_motion_planner:
                    act = mp.sample()
                else:
                    act = env.action_space.sample()

                obs, r, done, _, _ = env.step(act)

                rewards.append(float(r))        # scalar
                actions.append(int(act))        # scalar int

                if done:                        # keep fixed length anyway
                    # pad the remainder by repeating last frame / zeros reward / no-op
                    for _pad in range(t + 1, episode_length):
                        frames.append(obs)
                        rewards.append(0.0)
                        actions.append(0)
                    break

            # Stack & reshape
            # frames: list of (C,H,W) -> (T,C,H,W) -> (C,T,H,W), normalize to [0,1]
            frames = torch.stack(frames, dim=0).float()  # (T,C,H,W)
            frames = frames.permute(1,0,2,3).contiguous()  # (C,T,H,W)
            frames = frames.detach().cpu()  # <-- keep CPU


            rewards = torch.tensor(rewards, dtype=torch.float32)  # (T,)
            discrete = torch.tensor(actions, dtype=torch.long).unsqueeze(-1)  # (T,1)

            # sanity
            assert frames.shape[1] == episode_length
            assert rewards.shape[0] == episode_length
            assert discrete.shape[:2] == (episode_length, 1)

            self.samples.append({
                "video": frames,                # (C,T,H,W) float[0,1]
                "rewards": rewards,             # (T,)
                "discrete_actions": discrete,   # (T,1)
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]

        # support both storage styles
        vid_key = "video_uint8" if "video_uint8" in s else "video"
        v = s[vid_key]                 # (C,T,H,W)
        r = s["rewards"]               # (T,)
        a = s["discrete_actions"]      # (T,1)

        C, T, H, W = v.shape
        K = self.return_length
        if K > T:
            raise ValueError(f"return_length ({K}) > episode_length ({T})")

        t0 = random.randint(0, T - K)  # inclusive range
        t1 = t0 + K

        # slice time dimension correctly
        v = v[:, t0:t1]                # (C,K,H,W)
        r = r[t0:t1]                   # (K,)
        a = a[t0:t1]                   # (K,1)

        # normalize if stored as uint8
        if v.dtype == torch.uint8:
            v = v.float().div_(255.0)

        # normalize to lie between -1 and 1
        v = v * 2.0 - 1.0

        # keep memory friendly
        v = v.contiguous()
        r = r.contiguous()
        a = a.contiguous()

        return {"video": v, "rewards": r, "discrete_actions": a}


class TokFromBCEpisodes(Dataset):
    """
    Sliding windows over bc_ds.samples[*]["video"] without extra storage.
    Assumes bc_ds.samples[i]["video"] is (C,T,H,W) on CPU.
    """
    def __init__(self, bc_ds, window=16, stride=1):
        self.bc = bc_ds
        self.window = window
        self.stride = stride
        self.T = bc_ds.episode_length
        self.starts_per_ep = max(0, (self.T - window)//stride + 1)

    def __len__(self):
        return len(self.bc.samples) * self.starts_per_ep

    def __getitem__(self, i):
        ep_idx = i // self.starts_per_ep
        start  = (i % self.starts_per_ep) * self.stride
        v = self.bc.samples[ep_idx]["video"]          # (C,T,H,W) CPU
        v_window = v[:, start:start+self.window]      # (C,K,H,W) view

        # 2. FIX: Shift to [-1, 1] to match Codebase B expectations
        v_window = v_window * 2.0 - 1.0

        return v_window

def build_pinpad_datasets_for_trainers(
    n_episodes=200,
    episode_length=100,
    tok_window=16,
    device="cpu",
    use_motion_planner=True,
    normalize_in_env=True,   # <— new
):
    # base env
    base_env = PinPad('three', length=episode_length, extra_obs=False,
                      size=[64, 64], random_starting_pos=True, device=device)
    env = NormalizeObsWrapper(base_env) if normalize_in_env else base_env
    # ---- BC dataset (BehaviorCloneTrainer expects dict -> model(**batch))
    # Separate env so RNG/state differs a bit
    base_env_bc = PinPad('three', length=episode_length, extra_obs=False,
                         size=[64, 64], random_starting_pos=True, device=device)
    env_bc = NormalizeObsWrapper(base_env_bc) if normalize_in_env else base_env_bc
    bc_ds = PinPadBCEpisodes(env_bc, n_episodes=n_episodes,
                             episode_length=episode_length,
                             use_motion_planner=use_motion_planner,
                             return_length=tok_window)


    tok_ds = TokFromBCEpisodes(bc_ds, window=tok_window, stride=1)
    return tok_ds, bc_ds, env

In [ ]:
#| export 
# def dataset_to_experience(dataset):
_, bc_ds, _ = build_pinpad_datasets_for_trainers(10)


In [10]:
#| hide
import nbdev; nbdev.nbdev_export()